# Кластеризация по ценовому позиционированию с учётом фасовки

Вариант `03_clustering_prices_new_percentiles.ipynb` с двумя уточнениями:

1. **Один производитель полностью исключён** из анализа - у него отдельная модель ценообразования,
   не сопоставимая с остальным рынком, поэтому он искажал бы перцентили категорий (в портфолио-версии - `EXCLUDED_MANUFACTURER_FILE`, сам файл с кодами товаров не включён).
2. **Цена приводится к одной дозе/единице фасовки**, а не считается за упаковку целиком - иначе товар в
   упаковке по 100 таблеток будет выглядеть "дороже" товара в упаковке по 10, хотя по факту дешевле в пересчёте
   на дозу. Часть товаров с неоднозначной фасовкой из анализа исключается отдельно (справочник корректировок).

Дальше логика та же, что в предыдущем ноутбуке: перцентили цены-за-дозу внутри категории → метка low/medium/high
у товара → доля выручки точки по каждому уровню → кластеризация точек по этому профилю.

## Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import sqlalchemy

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import scipy.cluster.hierarchy as shc
from sklearn.cluster import AgglomerativeClustering
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap

import gc

pd.options.display.max_columns = 100

## Настройка параметров

In [ ]:
new_lb = pd.to_datetime('20221121')
new_rb = pd.to_datetime('20221123')

OPTICS_TRADE_POINT_TYPE_ID = '4'
EXCLUDED_AP_GROUPS = ['DD', 'D1', 'D2', 'D3', 'D4']
EXCLUDED_COUNTERPARTY_GROUP_ID = 8538

# справочники, специфичные для этого варианта (в портфолио не включены):
EXCLUDED_MANUFACTURER_FILE = 'исключённый_производитель_позиции.xlsx'  # товары одного производителя вне анализа
PACKAGE_SIZE_FILE = 'корректировка_фасовки.xlsx'  # число доз в упаковке + список товаров с неоднозначной фасовкой

PRICE_TIER_LOW_PERCENTILE = 30
PRICE_TIER_HIGH_PERCENTILE = 70

N_CLUSTERS = 7

## Функции

In [ ]:
def get_db_engine(name: str):
    """Заглушка вместо реального подключения к корпоративному DWH (SQL Server, ODBC) - см. 01_get_optics.ipynb."""
    raise NotImplementedError("Подключение к реальной БД в портфолио-версии не используется")


def batch_load_sql_dwh(query: str) -> pd.DataFrame:
    """Чанкованная загрузка результата SQL-запроса из DWH."""
    CHUNKSIZE = 200_000
    engine = get_db_engine('dwh')
    conn = engine.connect().execution_options(stream_results=True)
    chunks = []
    for chunk_dataframe in pd.read_sql(query, conn, chunksize=CHUNKSIZE):
        chunks.append(chunk_dataframe)
    conn.close()
    return pd.concat(chunks, ignore_index=True)

In [ ]:
def percentile(n):
    """Фабрика агрегирующих функций для groupby.agg - percentile(30) возвращает функцию 30-го перцентиля."""
    def percentile_(x):
        return np.percentile(x, n)
    percentile_.__name__ = f'percentile_{n}'
    return percentile_


def price_tier(row):
    """Относит товар к low/medium/high по тому, где его цена за дозу лежит между перцентилями категории."""
    if row['mean_package_per_dose'] < row['percentile_low']:
        return 'low'
    elif row['percentile_low'] <= row['mean_package_per_dose'] <= row['percentile_high']:
        return 'medium'
    else:
        return 'high'

## Алгоритм выполнения

### Отбор точек, товарных категорий и справочника фасовки

In [ ]:
trade_points = pd.read_sql(f"""
    SELECT t1.STORE_LOCATION_ID,
        SUBSTRING(t1.STORE_LOCATION_NM,1,6) as TradePointCode,
        t1.STORE_LOCATION_LVL_ID3,
        t2.PRODUCT_ATTR_NM,
        t1.STORE_LOCATION_LVL_NM1,
        t1.CLOSE_DTTM,
        t1.OPEN_DTTM,
        tp.TradeSquare,
        t1.STORE_LOCATION_NM
    FROM dwh.IN_STORE_LOCATION_NEW t1
    INNER JOIN dwh.IN_CLUSTER_BASKET t2 ON (t1.STORE_LOCATION_LVL_ID3 = t2.STORE_LOCATION_LVL_ID3)
    INNER JOIN dwh.TradePoints tp ON tp.TradePointId = t1.STORE_LOCATION_ID
    WHERE tp.TradePointTypeId != '{OPTICS_TRADE_POINT_TYPE_ID}'
    """, con=get_db_engine('sas_ia'))
trade_points['CLOSE_DTTM'] = pd.to_datetime(trade_points['CLOSE_DTTM'])

trade_points = trade_points.loc[(trade_points.CLOSE_DTTM > new_rb)]
trade_points = trade_points.astype({"CLOSE_DTTM": 'datetime64[ns]', "OPEN_DTTM": 'datetime64[ns]'})
trade_points = trade_points.assign(time_dur_to_now=lambda x: (new_rb - x.OPEN_DTTM) / np.timedelta64(1, 'M'))
trade_points = trade_points.query('time_dur_to_now >= 6')

trade_points_brands = pd.read_sql("""
SELECT t2.BrandName, t1.TradePointCode
FROM dwh.TradePoints t1
INNER JOIN dwh.TradePointBrands t2 ON t1.BrandId = t2.BrandId
""", con=get_db_engine('sas_ia'))

In [ ]:
ap_cat_filtr = pd.read_excel('Категории для кластеризации.xlsx', 'Категории для кластеризации')
ap_cat_filtr = ap_cat_filtr.loc[ap_cat_filtr['Категория для кластеризации'] != 'удалить из данных']
ap_cat_filtr = ap_cat_filtr.loc[ap_cat_filtr['для категорий и ценовой кластеризации'] != 'удалить из данных']
groups_to_keep = ap_cat_filtr['2-Группа']

groups_cluster = pd.read_excel('groups_cluster.xlsx')
ap_cat_filtr = ap_cat_filtr.rename({'2-Группа': 'Groups'}, axis=1)
groups_cluster = groups_cluster.merge(ap_cat_filtr[['Groups', 'для ценовой 2']], on='Groups', how='right')

clust_names = dict(zip(
    groups_cluster.loc[groups_cluster['для ценовой 2'] == 'Красота и гигиена'].Cluster,
    groups_cluster.loc[groups_cluster['для ценовой 2'] == 'Красота и гигиена']['для ценовой 2'],
))

### Чеки и остатки

Те же запросы и фильтры, что и в `03_clustering_prices_new_percentiles.ipynb`.

In [ ]:
df = batch_load_sql_dwh(f"""
SELECT
    TradePointId,
    fc.ProductId,
    ap.apCode,
    Date,
    Time,
    Sum_Fact,
    Amount,
    Sum_Purch,
    Sum_Price,
    rowNumber as ChequePos,
    ChequeHeadersId,
    voda.Groups,
    voda.Cluster
FROM dbo.Fact_cheque fc
  JOIN dbo.InternetOrders io ON fc.InternetOrderId = io.InternetOrderId
  JOIN dbo.InternetOrderTypes iot ON fc.InternetOrderTypeId = iot.InternetOrderTypeId
  JOIN dbo.Parts p ON fc.PartId = p.PartId
  JOIN dbo.Products prod ON prod.ProductId = fc.ProductId
  JOIN dwh.dimAP ap ON ap.apID = prod.apId
  JOIN dbo.v_OLAP_dimAP voda ON voda.apCode = ap.apCode
  JOIN dwh.DicCounterparty d ON fc.DicCustomerId = d.DicCounterpartyId
  JOIN dwh.Counterparties c ON d.CounterpartyId = c.CounterpartyId
  JOIN dwh.CounterpartyGroups g ON c.GroupId = g.GroupId
WHERE fc.IsInternalId = 0
  AND (fc.SaleTypeId = 1 OR fc.SaleTypeId = 3 OR fc.SaleTypeId = 4 OR fc.SaleTypeId = 5)
  AND io.InternetOrderSourceId NOT IN (6, 38)
  AND datediff(month, fc.[Date], p.ExpireDate) >= 6
  AND voda.Class != 'Нетоварные позиции'
  AND TradePointId IN {tuple(trade_points.STORE_LOCATION_ID)}
  AND (Date BETWEEN {"'" + str(new_lb.date()) + "'"} AND {"'" + str(new_rb.date()) + "'"})
  AND voda.Groups != 'UN'
  AND g.GroupID != {EXCLUDED_COUNTERPARTY_GROUP_ID}
""")
df = df.query('Groups in @groups_to_keep')

dd_groups = batch_load_sql_dwh(f"""SELECT apCode FROM dwh.dimAP da
JOIN dwh.dimAPGroups da2 ON da.apGroupID = da2.apGroupID
WHERE apGroupCode IN {tuple(EXCLUDED_AP_GROUPS)}
""")
df = df[~df.apCode.isin(list(dd_groups.apCode))]

Дополнительно (в отличие от предыдущего ноутбука) - исключаются все товары одного производителя.

In [ ]:
excluded_manufacturer = pd.read_excel(EXCLUDED_MANUFACTURER_FILE)
excluded_manufacturer = excluded_manufacturer.rename({'Код АП': 'apCode'}, axis=1).astype({'apCode': 'object'})
df = df[~df.apCode.isin(list(map(str, excluded_manufacturer.apCode)))]

df['Cluster'][df['Cluster'].isin(list(clust_names.keys()))] = 'Красота и гигиена'  # объединение мета-кластера

In [ ]:
df_stock = batch_load_sql_dwh(f"""
SELECT
    fsdm.ProductId, ap.apCode, Amount, Sum_Purch, Sum_Price, TradePointId, voda.Groups, voda.Cluster
FROM dbo.Fact_Stock_dataMart fsdm
JOIN dbo.DicWarehouse dw ON fsdm.DicWarehouseId = dw.DicWarehouseId
JOIN dbo.Warehouses w ON dw.Warehouse_GUID = w.Warehouse_GUID
JOIN dbo.DicWarehouseType dwt ON w.Type_GUID = dwt.WarehouseType_GUID
LEFT JOIN dbo.InternetOrders io ON fsdm.InternetOrderId = io.InternetOrderId
JOIN dbo.Products prod ON prod.ProductId = fsdm.ProductId
JOIN dwh.dimAP ap ON ap.apID = prod.apId
JOIN dbo.v_OLAP_dimAP voda ON voda.apCode = ap.apCode
JOIN dbo.Parts p ON fsdm.PartId = p.PartId
WHERE
    dwt.DicWarehouseTypeName = 'Торговая точка'
    AND isnull(io.InternetOrderSourceId, 1) NOT IN (6, 38)
    AND datediff(month, fsdm.[Date], p.ExpireDate) >= 6
    AND voda.Class != 'Нетоварные позиции'
    AND voda.Groups != 'UN'
    AND TradePointId IN {tuple(trade_points.STORE_LOCATION_ID)}
""")
df_stock = df_stock.query('Groups in @groups_to_keep')
df_stock = df_stock[~df_stock.apCode.isin(list(dd_groups.apCode))]
df_stock = df_stock[~df_stock.apCode.isin(list(map(str, excluded_manufacturer.apCode)))]
df_stock['Cluster'][df_stock['Cluster'].isin(list(clust_names.keys()))] = 'Красота и гигиена'

### Приведение цены к одной дозе (фасовке)

Справочник фасовки даёт итоговое число доз в упаковке для части товаров (для остальных считаем 1 - то есть
без коррекции). Отдельно из этого же справочника берётся список товаров с заведомо неоднозначной фасовкой -
их проще исключить из анализа, чем считать по ним искажённую цену за дозу.

In [ ]:
package_size = pd.read_excel(PACKAGE_SIZE_FILE, 'без дублей').rename(columns={'Код АП': 'apCode'})
package_size_exclude = pd.read_excel(PACKAGE_SIZE_FILE, 'фасовки').rename(columns={'Код АП': 'apCode'})
package_size_exclude = package_size_exclude.loc[package_size_exclude['исключить'] == 1]['apCode']

df_stock = df_stock[~df_stock.apCode.isin(list(map(str, package_size_exclude)))]
df = df[~df.apCode.isin(list(map(str, package_size_exclude)))]
package_size = package_size.astype({'apCode': 'str'})

### Средняя цена за дозу: остатки с приоритетом, чеки как fallback

In [ ]:
stock_avg = df_stock.groupby(['Cluster', 'apCode'], as_index=False).agg({'Sum_Purch': 'sum', 'Amount': 'sum'})
stock_avg.rename({'Sum_Purch': 'Revenue'}, axis=1, inplace=True)
stock_avg = stock_avg.assign(mean_package=lambda x: x.Revenue / x.Amount)
stock_avg = stock_avg[['Cluster', 'apCode', 'mean_package']]

stock_avg = stock_avg.merge(package_size[['apCode', 'Итоговая фасовка']], on='apCode', how='left')
stock_avg.loc[stock_avg['Итоговая фасовка'].isna(), 'Итоговая фасовка'] = 1  # нет данных по фасовке - считаем 1 доза
stock_avg['mean_package_per_dose'] = stock_avg.mean_package / stock_avg['Итоговая фасовка']

cheque_avg = df.groupby(['Cluster', 'apCode'], as_index=False).agg({'Sum_Purch': 'sum', 'Amount': 'sum'})
cheque_avg.rename({'Sum_Purch': 'Revenue'}, axis=1, inplace=True)
cheque_avg = cheque_avg.assign(mean_package=lambda x: x.Revenue / x.Amount)
cheque_avg = cheque_avg[['Cluster', 'apCode', 'mean_package']]

cheque_avg = cheque_avg.merge(package_size[['apCode', 'Итоговая фасовка']], on='apCode', how='left')
cheque_avg.loc[cheque_avg['Итоговая фасовка'].isna(), 'Итоговая фасовка'] = 1
cheque_avg['mean_package_per_dose'] = cheque_avg.mean_package / cheque_avg['Итоговая фасовка']

# по остаткам есть не для всех apCode - там, где не хватает, берём цену из чеков
price_avg = stock_avg.merge(cheque_avg, on='apCode', how='outer')
price_avg['mean_package_x'][price_avg['mean_package_x'].isna()] = price_avg['mean_package_y']
price_avg['Cluster_x'][price_avg['Cluster_x'].isna()] = price_avg['Cluster_y']
price_avg['mean_package_per_dose_x'][price_avg['mean_package_per_dose_x'].isna()] = price_avg['mean_package_per_dose_y']

price_avg = price_avg.drop(['mean_package_y', 'Cluster_y', 'mean_package_per_dose_y',
                             'Итоговая фасовка_x', 'Итоговая фасовка_y'], axis=1)
price_avg = price_avg.rename({'mean_package_x': 'mean_package', 'Cluster_x': 'Cluster',
                               'mean_package_per_dose_x': 'mean_package_per_dose'}, axis=1)
price_avg = price_avg.dropna()
price_avg

### Перцентили и разметка ценового уровня товара

In [ ]:
price_percentiles = price_avg.sort_values(by=['Cluster', 'mean_package_per_dose'])\
    .groupby(['Cluster'], as_index=False)\
    .agg({'mean_package_per_dose': [percentile(PRICE_TIER_LOW_PERCENTILE), percentile(PRICE_TIER_HIGH_PERCENTILE)]})
price_percentiles.columns = price_percentiles.columns.droplevel()
price_percentiles.columns = ['Cluster', 'percentile_low', 'percentile_high']

price_avg = price_avg.merge(price_percentiles, on='Cluster', how='left')
price_avg['price_category'] = price_avg.apply(price_tier, axis=1)
price_avg

Дополнительные бизнес-правила к границам категорий считаются здесь по **не скорректированной на фасовку**
цене (`mean_package`) - то есть "узость" ценового диапазона категории и её "дешевизна" оцениваются по номинальной
цене упаковки, а не по цене за дозу:

1. если верхний и нижний перцентиль почти совпадают (разброс < 20%) - категория целиком исключается из анализа;
2. если категория в целом дешёвая - весь товар в ней размечается как `low`.

In [ ]:
raw_percentiles = price_avg[['Cluster', 'apCode', 'mean_package']].sort_values(by=['Cluster', 'mean_package'])\
    .groupby(['Cluster'], as_index=False)\
    .agg({'mean_package': [percentile(PRICE_TIER_LOW_PERCENTILE), percentile(PRICE_TIER_HIGH_PERCENTILE)]})
raw_percentiles.columns = raw_percentiles.columns.droplevel()
raw_percentiles.columns = ['Cluster', 'percentile_low', 'percentile_high']
raw_percentiles = raw_percentiles.merge(price_avg[['Cluster', 'apCode']], on='Cluster', how='left')

# правило 1: слишком узкий ценовой разброс внутри категории (по номинальной цене) - категория исключается целиком
raw_percentiles = raw_percentiles.assign(too_narrow_range=((raw_percentiles.percentile_high / raw_percentiles.percentile_low) - 1) < 0.2)
ap_to_exclude = list(raw_percentiles.apCode[raw_percentiles.too_narrow_range == True])

# правило 2: категория в целом дешёвая - весь товар в ней размечается как low
raw_percentiles = raw_percentiles.assign(
    force_low=(raw_percentiles.percentile_high < 100)
    | ((raw_percentiles.percentile_high < 150) & (((raw_percentiles.percentile_high / raw_percentiles.percentile_low) - 1) < 1))
)
clusters_forced_low = raw_percentiles.Cluster[raw_percentiles['force_low'] == True]
price_avg.price_category[price_avg.Cluster.isin(clusters_forced_low)] = 'low'

### Доля выручки точки по ценовым уровням и кластеризация

In [ ]:
df = df[~df.apCode.isin(ap_to_exclude)]  # правило 1
df = df.merge(price_avg[['apCode', 'price_category']], how='left', on='apCode')
df = df.dropna()

revenue_by_tp = df.groupby('TradePointId', as_index=False).agg({'Sum_Purch': 'sum'})
revenue_by_tp.rename({'Sum_Purch': 'Revenue'}, axis=1, inplace=True)

revenue_by_tp_tier = df.groupby(['TradePointId', 'price_category'], as_index=False).agg({'Sum_Purch': 'sum'})
revenue_by_tp_tier.rename({'Sum_Purch': 'Revenue'}, axis=1, inplace=True)

revenue_by_tp_tier = revenue_by_tp_tier.merge(revenue_by_tp, on='TradePointId', how='left')
revenue_by_tp_tier = revenue_by_tp_tier.assign(revenue_share=revenue_by_tp_tier.Revenue_x / revenue_by_tp_tier.Revenue_y)

price_profile = revenue_by_tp_tier.pivot(index='TradePointId', columns='price_category', values='revenue_share')
price_profile

Кластеризация - тот же метод (Ward, 7 кластеров), что и в предыдущем ноутбуке.

In [ ]:
mpl.rc_file_defaults()
dend = shc.dendrogram(shc.linkage(price_profile, method='ward'),
                       truncate_mode='lastp', p=N_CLUSTERS, show_leaf_counts=True)

In [ ]:
cluster = AgglomerativeClustering(n_clusters=N_CLUSTERS, affinity='euclidean', linkage='ward')
clst_prices = cluster.fit_predict(price_profile)

pd.Series(clst_prices, name='ClusterId').value_counts()

In [ ]:
fig = plt.figure(figsize=(15, 9))
Z = shc.linkage(price_profile, method='ward')
last = Z[-68:, 2]
last_rev = last[::-1]
idxs = np.arange(1, len(last) + 1)
plt.plot(idxs, last_rev)

acceleration = np.diff(last, 2)
acceleration_rev = acceleration[::-1]
plt.plot(idxs[:-2] + 1, acceleration_rev)

plt.axvline(x=N_CLUSTERS, ymax=0.6, color='r', ls=':', lw=2)
plt.show()

In [ ]:
tsne_coords = TSNE(random_state=2023).fit_transform(price_profile)
tsne_coords = pd.DataFrame(tsne_coords).rename({0: 'x', 1: 'y'}, axis=1)
tsne_coords = tsne_coords.assign(ID_prices=pd.Series(clst_prices, name='ClusterId'))

PALETTE_7 = ['#808000', '#008000', '#00FF00', '#FF0000', '#b35806', '#FF8C00', '#48D1CC']
fig = plt.figure(figsize=(15, 9))
ax = sns.scatterplot(x='x', y='y', data=tsne_coords, hue='ID_prices', legend='full', palette=sns.color_palette(PALETTE_7))

In [ ]:
# отдельно - только 3 конкретных кластера, чтобы рассмотреть их разделимость крупнее
PALETTE_3 = ['#FF0000', '#008000', '#00FF00']
fig = plt.figure(figsize=(15, 9))
ax = sns.scatterplot(x='x', y='y',
                      data=tsne_coords.query('ID_prices == 4 or ID_prices == 0 or ID_prices == 3'),
                      hue='ID_prices', legend='full', palette=sns.color_palette(PALETTE_3))

In [ ]:
to_plot = price_profile.reset_index(drop=True).assign(ID_prices=pd.Series(clst_prices, name='ClusterId'))

fig = plt.figure(figsize=(6, 6))
ax = Axes3D(fig, auto_add_to_figure=False)
fig.add_axes(ax)

cmap = ListedColormap(sns.color_palette("husl", 256).as_hex())
sc = ax.scatter(to_plot.low, to_plot.medium, to_plot.high, s=40, c=to_plot.ID_prices, marker='o', cmap=cmap, alpha=1)
ax.set_xlabel('low')
ax.set_ylabel('medium')
ax.set_zlabel('high')
ax.azim = 20
ax.elev = 5
plt.legend(*sc.legend_elements(), bbox_to_anchor=(1.05, 1), loc=2)

### Сборка итоговой таблицы и визуализация

In [ ]:
result = price_profile.copy().reset_index()
result = result.assign(ID_prices=pd.Series(clst_prices, name='ClusterId'))

result = result.merge(trade_points[['STORE_LOCATION_ID', 'TradePointCode',
                                     'PRODUCT_ATTR_NM', 'STORE_LOCATION_LVL_NM1', 'STORE_LOCATION_NM']],
                       left_on='TradePointId', right_on='STORE_LOCATION_ID', how='left')\
    .drop(['STORE_LOCATION_ID'], axis=1)

result = result.merge(trade_points_brands, how='left', on='TradePointCode')
result = result.drop_duplicates('TradePointCode', keep='first')
result = result.drop(['PRODUCT_ATTR_NM', 'TradePointId'], axis=1)

# экспорт (не выполняется в портфолио-версии):
# result.to_excel("fassovka_prices_clustering_percentiles.xlsx", index=False)
# result.groupby('ID_prices').describe().to_excel("fassovka_prices_clustering_stats_percentiles.xlsx")
result

In [ ]:
revenue_and_cheques = df.groupby('TradePointId', as_index=False).agg({'Sum_Purch': 'sum', 'ChequeHeadersId': 'count'})
revenue_and_cheques.rename({'Sum_Purch': 'Revenue', 'ChequeHeadersId': 'ChequeCount'}, axis=1, inplace=True)
revenue_and_cheques = revenue_and_cheques.assign(mean_purch=lambda x: x.Revenue / x.ChequeCount)

revenue_and_cheques = revenue_and_cheques.merge(
    trade_points[['STORE_LOCATION_ID', 'TradePointCode']], left_on='TradePointId', right_on='STORE_LOCATION_ID', how='left'
).drop(['STORE_LOCATION_ID'], axis=1)

result = result.merge(revenue_and_cheques[['TradePointCode', 'Revenue', 'mean_purch']], how='left', on='TradePointCode')
result

In [ ]:
PALETTE_11 = ['#0000FF', '#800000', '#00FF00', '#000080', '#FF0000', '#FF8C00',
              '#48D1CC', '#4B0082', '#DC143C', '#808080', '#FF00FF']

fig = plt.figure(figsize=(17, 10))
sns.swarmplot(x='ID_prices', y='Revenue', hue='BrandName', order=[4, 0, 2, 3, 1, 6, 5], data=result, size=7,
              palette=PALETTE_11)
plt.show()

In [ ]:
fig = plt.figure(figsize=(17, 10))
sns.swarmplot(x='ID_prices', y='mean_purch', hue='BrandName', order=[4, 0, 2, 3, 1, 6, 5], data=result, size=7,
              palette=PALETTE_11)
plt.show()